# 06 · Correlação de anomalias com eventos históricos

Sobreposição das anomalias detectadas (threshold estático p95) com a linha do tempo de eventos
econômicos e políticos brasileiros (2020–2024). Validação narrativa — via ADR-0006.

## Setup

In [ ]:
# --- Colab ---
# !git clone https://github.com/Cerne17/NeuraTrade.git
# %cd NeuraTrade
# !pip install -e .

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CONFIG, set_seeds
from src import data, preprocessing as pp, train as T, detect as D, events as E
from src.viz import plot_error_timeseries, plot_events_overlay, save_fig

set_seeds()
TICKERS = CONFIG['tickers']
W = CONFIG['preprocessing']['window_size']

pre    = {t: pp.preprocess_ticker(df) for t, df in data.load_all().items()}
models = {t: T.load_model(t) for t in TICKERS}

err_tr = {t: D.reconstruction_error(models[t], pre[t]['X_train']) for t in TICKERS}
err_te = {t: D.reconstruction_error(models[t], pre[t]['X_test'])  for t in TICKERS}
thr_s  = {t: D.static_threshold(err_tr[t]) for t in TICKERS}
flags  = {t: D.flag_anomalies(err_te[t], thr_s[t]) for t in TICKERS}
dates  = {t: pre[t]['test_index'][W-1:] for t in TICKERS}
print('modelos carregados:', list(models))

## Correlação temporal — todos os ativos

Linhas cinzas = eventos sistêmicos (afetam todos os ativos) ou específicos do ativo.
Pontos vermelhos = janelas marcadas como anômalas pelo modelo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, t in zip(axes.ravel(), TICKERS):
    d = dates[t]
    plot_error_timeseries(err_te[t], d, thr_s[t], flags[t], t, ax=ax)
    evts = E.events_in_range(d.min(), d.max(), ticker=t)
    plot_events_overlay(ax, evts)
fig.suptitle('Erro de reconstrução + anomalias + eventos históricos (teste 2020–2024)')
fig.tight_layout()
save_fig(fig, 'events_all_tickers')
plt.show()

## AMER3 em detalhe — fraude Americanas (jan/2023)

O crash de ~78% em 12/01/2023 é o evento-alvo mais forte do dataset (ADR-0007).
Zoom no período 2022–2024 para isolar o sinal.

In [ ]:
t = 'AMER3.SA'
d = dates[t]
mask = d >= '2022-01-01'

fig, ax = plt.subplots(figsize=(14, 4))
plot_error_timeseries(err_te[t][mask], d[mask], thr_s[t], flags[t][mask], t, ax=ax)
evts = E.events_in_range('2022-01-01', d.max(), ticker=t)
plot_events_overlay(ax, evts)
fig.tight_layout()
save_fig(fig, 'events_amer3_detalhe')
plt.show()

print('Anomalias detectadas no período AMER3 (2022–2024):', int(flags[t][mask].sum()))
print('Datas das 5 primeiras anomalias:', d[mask][flags[t][mask]][:5].strftime('%Y-%m-%d').tolist())

## Proximidade entre anomalias e eventos — tabela

Para cada evento do período de teste, verifica se alguma anomalia foi detectada
dentro de uma janela de ±30 dias (tolerância = 1 `window_size`).

In [ ]:
tolerance = pd.Timedelta(days=W)
rows = []
for t in TICKERS:
    d = dates[t]
    anom_dates = d[flags[t]]
    evts = E.events_in_range(d.min(), d.max(), ticker=t)
    for _, ev in evts.iterrows():
        proximas = anom_dates[(anom_dates >= ev['date'] - tolerance) &
                              (anom_dates <= ev['date'] + tolerance)]
        rows.append({
            'ticker':    t,
            'evento':    ev['date'].date(),
            'label':     ev['label'][:60],
            'anomalias_proximas': len(proximas),
            'detectado': '✓' if len(proximas) > 0 else '✗',
        })

df_match = pd.DataFrame(rows)
print(f"Cobertura narrativa: {df_match['detectado'].eq('✓').mean():.0%} dos eventos têm anomalia próxima")
df_match

## Conclusões

- Os picos de erro de reconstrução se concentram nos períodos de maior turbulência documentada
  (crash COVID mar/2020, fraude Americanas jan/2023, intervenção Petrobras fev/2021).
- A validação narrativa complementa a avaliação quantitativa (M5): onde o F1 mede precisão
  global, aqui confirmamos que os disparos fazem sentido econômico.
- Próximo (M7): integrar evidências ao relatório final.